# Insert test record - Price DQ rule violation

Inserts **one** product into `dbo.Product` that fails the rule **"List price is non-negative"** (Validity, High) and passes every other rule:

* `Price` is negative, so the price rule fails.
* `ID` and `Name` are populated, so the completeness rules pass.

Expected result after running `02_evaluate_product_dq_rules`: one failed rule row in `dq.dq_rule_result` for this product, and its `dq_score` drops by that rule's weight (25 points).

* Attach this notebook to the Lakehouse that holds the product table.
* Idempotent: re-running updates the same test row (MERGE on `ID`) instead of duplicating it.
* Use the **Cleanup** cell at the end to remove the test row.

In [ ]:
# ---- Config ----------------------------------------------------------------
SCHEMA_NAME   = "dbo"
TABLE_NAME    = "product"
TEST_ID       = "DQ-TEST-PRICE-0001"   # fixed ID so the record is easy to find and remove
INVALID_PRICE = -49.99                  # violates: Price IS NOT NULL AND Price >= 0

## 1. Insert (or refresh) the test product

In [ ]:
src = f"{SCHEMA_NAME}.{TABLE_NAME}"
assert spark.catalog.tableExists(src), f"{src} not found in the attached lakehouse"

spark.sql(f"""
MERGE INTO {src} AS t
USING (
    SELECT
        '{TEST_ID}'                          AS ID,
        'DQ Test - Negative Price Product'   AS Name,
        'DQ-TEST-SKU-0001'                   AS SKU,
        'Test record that violates the non-negative price DQ rule' AS Description,
        CAST({INVALID_PRICE} AS DECIMAL(34,6)) AS Price,
        10                                   AS StockQuantity,
        5                                    AS ReorderLevel,
        current_timestamp()                  AS CreatedDate,
        current_timestamp()                  AS ModifiedDate,
        true                                 AS IsActive,
        current_timestamp()                  AS CreatedOn,
        current_timestamp()                  AS ModifiedOn,
        'dq-test'                            AS CreatedByID,
        'dq-test'                            AS ModifiedByID,
        CAST(NULL AS STRING)                 AS StatusID,
        CAST(NULL AS STRING)                 AS OwnerID,
        'dq-test-negative-price-product'     AS URLSlug
) AS s
ON t.ID = s.ID
WHEN MATCHED THEN UPDATE SET
    t.Price = s.Price,
    t.ModifiedDate = s.ModifiedDate,
    t.ModifiedOn = s.ModifiedOn
WHEN NOT MATCHED THEN INSERT *
""")
print(f"Test product {TEST_ID} upserted into {src} with Price = {INVALID_PRICE}")

## 2. Verify the record violates the price rule

In [ ]:
display(spark.sql(f"""
    SELECT ID, Name, SKU, Price,
           (Price IS NOT NULL AND Price >= 0) AS passes_price_rule
    FROM {src}
    WHERE ID = '{TEST_ID}'
"""))

## 3. Cleanup (optional)

Uncomment and run to remove the test product once you have checked the DQ results.

In [ ]:
# spark.sql(f"DELETE FROM {src} WHERE ID = '{TEST_ID}'")
# print(f"Removed {TEST_ID} from {src}")